In [1]:
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer


In [2]:
!python -m spacy download en_core_web_lg
!python -m spacy download pt_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 22.0 MB/s  0:00:16:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 56.7 MB/s  0:00:00 eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')


In [3]:
motor_analise = AnalyzerEngine()

texto_exemplo = "My name is Jonh Lennon and my number is +1 305 555-1234"

resultados = motor_analise.analyze(
        text=texto_exemplo,
        language="en"
)

print(resultados)

[type: PERSON, start: 11, end: 22, score: 0.85, type: PHONE_NUMBER, start: 40, end: 55, score: 0.75]


In [11]:
analyzer_engine = AnalyzerEngine()

ticket_pattern = Pattern(
    name="support_ticket_rule",
    regex="TICKET-[0-9]{4}",
    score=0.9
)

custom_recognizer = PatternRecognizer(
    supported_entity="SUPPORT_TICKET",
    patterns=[ticket_pattern],
    supported_language="en"
)

analyzer_engine.registry.add_recognizer(custom_recognizer)

sample_text = "The customer reported an access failure in TICKET-4598 this morning."
results = analyzer_engine.analyze(text=sample_text, language="en")

print(results)

[type: SUPPORT_TICKET, start: 43, end: 54, score: 0.9]


In [8]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

analyzer_engine = AnalyzerEngine()
anonymizer_engine = AnonymizerEngine()

sample_text = "My phone number is +1 305 555-1234."
results = analyzer_engine.analyze(text=sample_text, language="en")

# Masking parameters must be passed as a dictionary
operators_config = {
    "PHONE_NUMBER": OperatorConfig(
        "mask", {"chars_to_mask": 4, "masking_char": "*", "from_end": True}
    )
}

safe_text = anonymizer_engine.anonymize(
    text=sample_text,
    analyzer_results=results,
    operators=operators_config
)

print(safe_text.text)

My phone number is +1 305 555-****.


In [5]:
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_analyzer.nlp_engine import NlpEngineProvider

# 1. Configura o motor NLP para suportar português
configuracao_nlp = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "pt", "model_name": "pt_core_news_sm"},
    ],
}

provedor = NlpEngineProvider(nlp_configuration=configuracao_nlp)
nlp_engine_pt = provedor.create_engine()

# 2. Inicializa o AnalyzerEngine informando o motor e os idiomas suportados
motor_analise = AnalyzerEngine(
    nlp_engine=nlp_engine_pt,
    supported_languages=["pt"]
)

# 3. Cria e registra seu reconhecedor customizado
molde_ticket = Pattern(
    name="regra_ticket_suporte",
    regex="TICKET-[0-9]{4}",
    score=0.9
)

reconhecedor_customizado = PatternRecognizer(
    supported_entity="TICKET_ATENDIMENTO",
    patterns=[molde_ticket],
    supported_language="pt"
)

motor_analise.registry.add_recognizer(reconhecedor_customizado)

# 4. Executa a análise
texto_exemplo = "O cliente relatou uma falha de acesso no TICKET-4598 hoje de manhã."
resultados = motor_analise.analyze(text=texto_exemplo, language="pt")

print(resultados)

[type: TICKET_ATENDIMENTO, start: 41, end: 52, score: 0.9]
